# Combinación de dengue clásico (210) + dengue grave (220)

Une los dos CSVs consolidados en un único dataset, añade la columna `tipo_dengue` y deduplica por `CONSECUTIVE` en caso de que haya registros que aparezcan en ambos archivos.

**Prerrequisito:** haber corrido los notebooks `02_Combinar_SIVIGILA_Dengue.ipynb` y `03_Combinar_SIVIGILA_Dengue_Grave.ipynb`.

**Salida:** `data/processed/sivigila_dengue_completo.csv`

In [ ]:
import os
import pandas as pd

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CSV_210    = "../data/processed/sivigila_dengue_consolidado.csv"
CSV_220    = "../data/processed/sivigila_dengue_grave_consolidado.csv"
OUTPUT_CSV = "../data/processed/sivigila_dengue_completo.csv"

In [ ]:
print("Cargando dengue clásico (210)...")
df_210 = pd.read_csv(CSV_210, dtype=str, low_memory=False)
df_210.insert(1, "tipo_dengue", "clasico")
print(f"  {len(df_210):,} filas, {df_210.shape[1]} columnas")

print("Cargando dengue grave (220)...")
df_220 = pd.read_csv(CSV_220, dtype=str, low_memory=False)
df_220.insert(1, "tipo_dengue", "grave")
print(f"  {len(df_220):,} filas, {df_220.shape[1]} columnas")

In [ ]:
# Verificar que comparten las mismas columnas (excluida tipo_dengue)
cols_210 = set(df_210.columns) - {"tipo_dengue"}
cols_220 = set(df_220.columns) - {"tipo_dengue"}

solo_en_210 = cols_210 - cols_220
solo_en_220 = cols_220 - cols_210

if not solo_en_210 and not solo_en_220:
    print("OK: Columnas identicas en ambos datasets")
else:
    if solo_en_210:
        print(f"Solo en 210: {solo_en_210}")
    if solo_en_220:
        print(f"Solo en 220: {solo_en_220}")

In [ ]:
# Detectar duplicados por CONSECUTIVE antes de concatenar
consec_210 = set(df_210["CONSECUTIVE"].dropna())
consec_220 = set(df_220["CONSECUTIVE"].dropna())
duplicados  = consec_210 & consec_220

print(f"CONSECUTIVE únicos en 210: {len(consec_210):,}")
print(f"CONSECUTIVE únicos en 220: {len(consec_220):,}")
print(f"Intersección:              {len(duplicados):,}")

if duplicados:
    pct = len(duplicados) / len(consec_220) * 100
    print(f"  → {pct:.1f}% de los graves están también en el dataset clásico")
    print(f"  Ejemplos: {list(duplicados)[:5]}")
else:
    print("  → Sin duplicados: los datasets son completamente independientes")

In [ ]:
# Concatenar
df_completo = pd.concat([df_210, df_220], ignore_index=True)
print(f"Total antes de deduplicar: {len(df_completo):,} filas")

# Deduplicar: si un CONSECUTIVE aparece en ambos, conservar la versión de 220
# (el registro grave tiene más información clínica)
if duplicados:
    # Ordenar para que grave quede primero, luego drop_duplicates conserva ese
    df_completo["_orden"] = df_completo["tipo_dengue"].map({"grave": 0, "clasico": 1})
    df_completo = (
        df_completo
        .sort_values("_orden")
        .drop_duplicates(subset="CONSECUTIVE", keep="first")
        .drop(columns="_orden")
        .reset_index(drop=True)
    )
    print(f"Total después de deduplicar: {len(df_completo):,} filas ({len(df_210)+len(df_220)-len(df_completo):,} duplicados eliminados)")
else:
    print("No hubo duplicados, no se aplicó deduplicación")

In [ ]:
# Resumen por tipo y año
resumen = (
    df_completo
    .groupby(["source_file", "tipo_dengue"])
    .size()
    .unstack(fill_value=0)
    .rename_axis("año")
    .reset_index()
)
resumen["total"] = resumen.get("clasico", 0) + resumen.get("grave", 0)
display(resumen.style.format({c: "{:,}" for c in resumen.columns if c != "año"}))

In [ ]:
# Guardar
os.makedirs(os.path.dirname(os.path.abspath(OUTPUT_CSV)), exist_ok=True)
df_completo.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"CSV guardado en: {os.path.abspath(OUTPUT_CSV)}")
print(f"Tamaño: {os.path.getsize(OUTPUT_CSV)/1_048_576:.0f} MB")

In [ ]:
# Verificación final
df_check = pd.read_csv(OUTPUT_CSV, nrows=3)
display(df_check[["source_file", "tipo_dengue", "CONSECUTIVE", "COD_EVE", "ANO", "Municipio_ocurrencia", "TIP_CAS"]])